# 02 — Dataset Development: leakage-safe note snapshots per coverage

Builds one dataset per target (**Attorney**, **Litigation**, **Severity**) with one row per
`(CLM_CVG_PK, snapshot)`: the claim's notes as they stood on a given day after the claim opened.

**Pseudocode**

```
for each closed coverage:
    boundary = event date if the event happened (attorney / litigation), else coverage close date
               (severity: last close date)
    keep only notes dated strictly before the boundary
    if no note falls exactly on the boundary and the last kept note mentions the event
        (keyword check), drop that note too

    for each day d in the target's day grid:
        cutoff = claim open date + d
        stop once cutoff reaches the boundary
        notes_so_far = kept notes dated before cutoff
        skip if notes_so_far is empty or unchanged since the previous snapshot
        write row: cumulative note text, number of notes, day d, target (constant per coverage)

split claims (not coverages) once into train / val / holdout (70/15/15), stratified on
    org group, policy state, loss year and all three targets; reuse this split for every target
write one parquet folder per (target, split)
```

**Key design choices**
- **Closed coverages only**, so every target is final.
- **Split by claim**: sibling coverages share the same notes, so they must land in the same split.
- **One global split** for all three targets and both feature strategies, so results are comparable.
- **Day grids** are denser early for Attorney (median 31 days to an attorney) and spread out for
  Litigation / Severity (median 666 days to a suit), based on `01`.
- A coverage can end up with no rows at all (claim has no notes, or every note is after the boundary).

Output: `{S3_BUCKET}/{SNAPSHOT_PREFIX}/{model}/{split}/*.parquet`


In [1]:
import glob
import os
import re
import html

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

from config import TARGET_DIR, NOTES_DIR, S3_BUCKET, TMP_DIR, SNAPSHOT_PREFIX as S3_PREFIX   # paths: config.py

DAY_GRIDS = {
    'Attorney':   [1, 7, 15, 30, 60, 90, 180, 365, 730, 1095],
    'Litigation': [1, 90, 180, 270, 365, 540, 730, 810, 900, 1095, 1460],
    'Severity':   [1, 90, 180, 270, 365, 540, 730, 810, 900, 1095, 1460, 1825],
}


def clean_note(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', '', text)
    text = html.unescape(text)
    return text

## Load base tables, build the closed-coverage universe + per-model boundary dates

In [2]:
sample = pd.read_parquet(os.path.join(TARGET_DIR, 'BI-Coverage-ALL-2015_2022.parquet'))  # reads local data
sev    = pd.read_parquet(os.path.join(TARGET_DIR, 'Endpoint-Severity-2015_2022.parquet'))  # reads local data

cov = sample[['CLM_NUM', 'CLM_CVG_PK', 'CLM_OPEN_DT', 'CVG_CLOSE_DT', 'CVG_STATUS_LOCKDOWN',
              'ORG_GROUP_NAME', 'POL_STATE', 'LOSS_YEAR',
              'ATTNY_FLG', 'ATTNY_DT', 'LITIG_FLG', 'LITIG_DT']].merge(
    sev[['CLM_CVG_PK', 'SEV_BUCKET', 'LAST_CLOSE_DT']], on='CLM_CVG_PK', how='left'
)

# closed coverages only
closed = cov[cov['CVG_CLOSE_DT'].notna()].copy()

# per-model boundary date
closed['BOUNDARY_ATTORNEY']   = closed['ATTNY_DT'].where(closed['ATTNY_FLG'] == 1, closed['CVG_CLOSE_DT'])
closed['BOUNDARY_LITIGATION'] = closed['LITIG_DT'].where(closed['LITIG_FLG'] == 1, closed['CVG_CLOSE_DT'])
closed['BOUNDARY_SEVERITY']   = closed['LAST_CLOSE_DT']

print(f'{len(cov):,} total coverages -> {len(closed):,} closed coverages in scope')
print(f'missing LAST_CLOSE_DT (would drop from Severity only): {closed.BOUNDARY_SEVERITY.isna().sum():,}')
closed.head()

169,322 total coverages -> 167,726 closed coverages in scope
missing LAST_CLOSE_DT (would drop from Severity only): 4,307


,CLM_NUM,CLM_CVG_PK,CLM_OPEN_DT,CVG_CLOSE_DT,CVG_STATUS_LOCKDOWN,ORG_GROUP_NAME,POL_STATE,LOSS_YEAR,ATTNY_FLG,ATTNY_DT,LITIG_FLG,LITIG_DT,SEV_BUCKET,LAST_CLOSE_DT,BOUNDARY_ATTORNEY,BOUNDARY_LITIGATION,BOUNDARY_SEVERITY
0,015188447294,29331908,2015-10-28,2017-04-10,CLOSED,Direct,NJ,2015,1,2015-11-10,0,NaT,1,2017-08-17,2015-11-10,2017-04-10,2017-08-17
1,070753645568,35877188,2015-06-15,2017-11-16,CLOSED,Direct,NJ,2015,1,2015-07-28,0,NaT,2,2017-11-15,2015-07-28,2017-11-16,2017-11-15
2,322646861642,39841622,2015-11-18,2017-10-16,CLOSED,Independent,NJ,2015,1,2016-05-06,0,NaT,0,2017-10-13,2016-05-06,2017-10-16,2017-10-13
3,181967273177,62477647,2015-02-26,2017-12-21,CLOSED,Independent,NJ,2015,1,2015-03-16,0,NaT,3,2017-12-21,2015-03-16,2017-12-21,2017-12-21
4,879922983695,83702838,2015-01-03,2017-12-06,CLOSED,Direct,NJ,2015,1,2015-02-26,0,NaT,2,2017-12-05,2015-02-26,2017-12-06,2017-12-05


## Margin-rule note filter

`filter_eligible(note_dates, boundary)` — strict `<`. If any note date exactly equals `boundary`,
plain `<` already excludes it correctly. If **no** note matches `boundary` exactly, drop the last
still-included note too (one extra note of safety margin).

In [3]:
KEYWORDS = {
    'Attorney':   re.compile(r'(?i)\batty\b|attorney|lawyer|\bLOR\b|letter of rep|represent|retain|\besq\b|counsel'),
    'Litigation': re.compile(r'(?i)summons|complaint|lawsuit|litigation|\bS&C\b|docket|subpoena|\bserved\b|\bsuit\b'),
    'Severity':   re.compile(r'(?i)settle|settlement|release|closing papers|payment issued|\bclosed\b|resolved|\bstip\b|stipulation'),
}


def eligible_note_count(claim_notes_sorted, boundary, model):
    """claim_notes_sorted: DataFrame with DATE + TEXT_NOTE columns, sorted by DATE, for one claim.
    boundary: the pd.Timestamp cutoff for this model.
    If no note exactly matches the boundary, only drop the last still-included note if its own text
    contains event-related language for this model (keyword search) -- otherwise keep it.
    """
    dates = claim_notes_sorted['DATE']
    n_before = int((dates < boundary).sum())
    if n_before == 0:
        return 0
    has_exact = bool((dates == boundary).any())
    if has_exact:
        return n_before
    candidate_text = clean_note(claim_notes_sorted.iloc[n_before - 1]['TEXT_NOTE'])
    if KEYWORDS[model].search(candidate_text):
        return n_before - 1
    return n_before


# quick unit check: same worked example, now with text column (no keywords -> should NOT drop)
_test_df = pd.DataFrame({
    'DATE': pd.to_datetime(['2020-01-10', '2020-01-20', '2020-01-30', '2020-02-09']),
    'TEXT_NOTE': ['routine note', 'another routine note', 'yet another note', 'final note'],
}).sort_values('DATE').reset_index(drop=True)

case_a = eligible_note_count(_test_df, pd.Timestamp('2020-01-30'), 'Attorney')
case_b_clean = eligible_note_count(_test_df, pd.Timestamp('2020-01-25'), 'Attorney')

_test_df_kw = _test_df.copy()
_test_df_kw.loc[1, 'TEXT_NOTE'] = 'LOR now in file, attorney retained'
case_b_keyword = eligible_note_count(_test_df_kw, pd.Timestamp('2020-01-25'), 'Attorney')

print('Case A (exact match @ day30) -> eligible count:', case_a, '(expected 2)')
print('Case B, no keyword in candidate note -> eligible count:', case_b_clean, '(expected 2, KEPT)')
print('Case B, keyword present in candidate note -> eligible count:', case_b_keyword, '(expected 1, DROPPED)')

Case A (exact match @ day30) -> eligible count: 2 (expected 2)
Case B, no keyword in candidate note -> eligible count: 2 (expected 2, KEPT)
Case B, keyword present in candidate note -> eligible count: 1 (expected 1, DROPPED)


## Core snapshot builder for one coverage

`build_snapshots(cvg_row, model, claim_notes)` — `claim_notes` is that CVG's claim's notes
(already sorted by date). Walks the model's day-grid, applies the eligible-note cap from the margin
rule, dedups unchanged note-counts, skips 0-note snapshots, and builds the cumulative `NOTE_TEXT`
with `SNAPSHOT (k)` + per-note day-gap formatting. Returns a list of row-dicts (possibly empty).

In [4]:
TARGET_COL = {'Attorney': 'TARGET', 'Litigation': 'TARGET', 'Severity': 'TARGET_BUCKET'}
BOUNDARY_COL = {'Attorney': 'BOUNDARY_ATTORNEY', 'Litigation': 'BOUNDARY_LITIGATION', 'Severity': 'BOUNDARY_SEVERITY'}
TARGET_SRC_COL = {'Attorney': 'ATTNY_FLG', 'Litigation': 'LITIG_FLG', 'Severity': 'SEV_BUCKET'}


def build_snapshots(cvg_row, model, claim_notes):
    boundary = cvg_row[BOUNDARY_COL[model]]
    if pd.isna(boundary):
        return []

    max_eligible_n = eligible_note_count(claim_notes, boundary, model)
    if max_eligible_n == 0:
        return []

    eligible = claim_notes.iloc[:max_eligible_n].reset_index(drop=True)
    open_dt = cvg_row['CLM_OPEN_DT']
    target_val = cvg_row[TARGET_SRC_COL[model]]

    rows = []
    note_text = ''
    prev_n = 0
    snapshot_num = 0

    for day in DAY_GRIDS[model]:
        cutoff = open_dt + pd.Timedelta(days=day)
        if cutoff >= boundary:
            break

        n_this = int((eligible['DATE'] < cutoff).sum())
        if n_this == 0 or n_this == prev_n:
            continue

        new_notes = eligible.iloc[prev_n:n_this]
        snapshot_num += 1

        if snapshot_num == 1:
            note_text += f"{'=' * 60}\nSNAPSHOT (1)\n{'=' * 60}\n\n"
        else:
            gap = (new_notes.iloc[0]['DATE'] - eligible.iloc[prev_n - 1]['DATE']).days
            note_text += (f"\n{'=' * 60}\nSNAPSHOT ({snapshot_num})  --  {gap} days after the "
                          f"previous snapshot's last note\n{'=' * 60}\n\n")

        for local_i in range(len(new_notes)):
            global_i = prev_n + local_i
            note_row = new_notes.iloc[local_i]
            if global_i == 0:
                gap_str = ''
            else:
                prev_date = eligible.iloc[global_i - 1]['DATE']
                gap_str = f'  |  +{(note_row["DATE"] - prev_date).days} days since previous note'
            note_text += f"{'=' * 60}\nNOTE {global_i + 1}  |  DATE: {note_row['DATE']}{gap_str}\n{'=' * 60}\n\n"
            note_text += clean_note(note_row['TEXT_NOTE'])
            note_text += '\n\n'

        rows.append({
            'CLM_CVG_PK': cvg_row['CLM_CVG_PK'],
            'CLM_NUM': cvg_row['CLM_NUM'],
            'SNAPSHOT_NUM': snapshot_num,
            'SNAPSHOT_DAY': day,
            'N_NOTES': n_this,
            'NOTE_TEXT': note_text,
            TARGET_COL[model]: target_val,
        })
        prev_n = n_this

    return rows

## Small-sample test — before running on everything

Pick a handful of closed CVGs that we know have decent note activity, run `build_snapshots` for the
Attorney model, and print the result to sanity-check the formatting.

In [5]:
TEST_CLM_NUM = '948648876634'  # a 19-note claim with attorney + litigation, used as a known example

test_cvg_rows = closed[closed['CLM_NUM'] == TEST_CLM_NUM]
print(f'{len(test_cvg_rows)} closed coverage(s) found for claim {TEST_CLM_NUM}')
display(test_cvg_rows[['CLM_CVG_PK', 'CLM_OPEN_DT', 'ATTNY_FLG', 'ATTNY_DT', 'LITIG_FLG', 'LITIG_DT',
                        'SEV_BUCKET', 'BOUNDARY_ATTORNEY', 'BOUNDARY_LITIGATION', 'BOUNDARY_SEVERITY']])

note_files = sorted(glob.glob(os.path.join(NOTES_DIR, 'batch_*.csv')))  # reads local data
rows = []
for f in note_files:
    df = pd.read_csv(f, usecols=['CLAIM_NUM', 'DATE', 'TEXT_NOTE'], dtype={'CLAIM_NUM': str})  # reads local data
    rows.append(df[df.CLAIM_NUM == TEST_CLM_NUM])

test_notes = pd.concat(rows, ignore_index=True)
test_notes['DATE'] = pd.to_datetime(test_notes['DATE'], errors='coerce')
test_notes = test_notes.sort_values('DATE').reset_index(drop=True)
print(f'\n{len(test_notes)} notes found for this claim')

1 closed coverage(s) found for claim 948648876634


,CLM_CVG_PK,CLM_OPEN_DT,ATTNY_FLG,ATTNY_DT,LITIG_FLG,LITIG_DT,SEV_BUCKET,BOUNDARY_ATTORNEY,BOUNDARY_LITIGATION,BOUNDARY_SEVERITY
72562,93194893,2018-08-27,1,2018-08-28,1,2018-08-27,1,2018-08-28,2018-08-27,2019-04-11



19 notes found for this claim


In [6]:
test_row = test_cvg_rows.iloc[0]

for model in ['Attorney', 'Litigation', 'Severity']:
    result = build_snapshots(test_row, model, test_notes)
    print(f'{model}: {len(result)} snapshot row(s)')
    for r in result:
        print(f"  snapshot {r['SNAPSHOT_NUM']} (day {r['SNAPSHOT_DAY']}): N_NOTES={r['N_NOTES']}")
    print()

Attorney: 0 snapshot row(s)

Litigation: 0 snapshot row(s)

Severity: 3 snapshot row(s)
  snapshot 1 (day 1): N_NOTES=1
  snapshot 2 (day 90): N_NOTES=3
  snapshot 3 (day 180): N_NOTES=9



## Broader test — 10 diverse real cases

Covers: sibling coverages of one claim with divergent outcomes, long attorney/litigation lags
(multi-snapshot), `ATTNY_FLG`/`LITIG_FLG` = 0 (fallback-to-close-date boundary), and a high note-count
claim (to exercise dedup more).

In [8]:
atty_lag  = (closed['ATTNY_DT'] - closed['CLM_OPEN_DT']).dt.days
litig_lag = (closed['LITIG_DT'] - closed['CLM_OPEN_DT']).dt.days

# 1) sibling coverages of one claim with divergent outcomes (ATTNY_FLG or SEV_BUCKET differ)
sib_counts = closed.groupby('CLM_NUM').size()
multi_claims = sib_counts[sib_counts >= 2].index
sib_claim = None
for cn in multi_claims:
    rows_ = closed[closed.CLM_NUM == cn]
    if rows_['SEV_BUCKET'].nunique() > 1 or rows_['ATTNY_FLG'].nunique() > 1:
        sib_claim = cn
        break
sib_rows = closed[closed.CLM_NUM == sib_claim]

long_atty  = closed[(closed.ATTNY_FLG == 1) & (atty_lag > 365)].sample(2, random_state=1)
long_litig = closed[(closed.LITIG_FLG == 1) & (litig_lag > 365)].sample(2, random_state=2)
no_atty    = closed[closed.ATTNY_FLG == 0].sample(2, random_state=3)
no_litig   = closed[closed.LITIG_FLG == 0].sample(2, random_state=4)

test_cases = []
for _, r in sib_rows.iterrows():
    test_cases.append((r, 'sibling-divergent'))
for _, r in long_atty.iterrows():
    test_cases.append((r, 'long-attorney-lag'))
for _, r in long_litig.iterrows():
    test_cases.append((r, 'long-litigation-lag'))
for _, r in no_atty.iterrows():
    test_cases.append((r, 'never-attorney'))
for _, r in no_litig.iterrows():
    test_cases.append((r, 'never-litigation'))

print(f'{len(test_cases)} test cases selected (sibling claim: {sib_claim})')
for r, label in test_cases:
    print(f"  [{label:22s}] CVG={r.CLM_CVG_PK}  CLM_NUM={r.CLM_NUM}  "
          f"ATTNY_FLG={r.ATTNY_FLG}  LITIG_FLG={r.LITIG_FLG}  SEV_BUCKET={r.SEV_BUCKET}")

# batch-load notes for all distinct claim nums involved, once
test_claim_nums = set(r.CLM_NUM for r, _ in test_cases)
rows = []
for f in note_files:
    df = pd.read_csv(f, usecols=['CLAIM_NUM', 'DATE', 'TEXT_NOTE'], dtype={'CLAIM_NUM': str})  # reads local data
    rows.append(df[df.CLAIM_NUM.isin(test_claim_nums)])
test_notes_all = pd.concat(rows, ignore_index=True)
test_notes_all['DATE'] = pd.to_datetime(test_notes_all['DATE'], errors='coerce')
test_notes_all = test_notes_all.sort_values(['CLAIM_NUM', 'DATE']).reset_index(drop=True)
print(f'\nLoaded notes for {test_notes_all.CLAIM_NUM.nunique()} distinct claims')

12 test cases selected (sibling claim: 015881399869)
  [sibling-divergent     ] CVG=64914524  CLM_NUM=015881399869  ATTNY_FLG=0  LITIG_FLG=0  SEV_BUCKET=0
  [sibling-divergent     ] CVG=72105218  CLM_NUM=015881399869  ATTNY_FLG=1  LITIG_FLG=1  SEV_BUCKET=3
  [sibling-divergent     ] CVG=51064600  CLM_NUM=015881399869  ATTNY_FLG=0  LITIG_FLG=0  SEV_BUCKET=0
  [sibling-divergent     ] CVG=23041189  CLM_NUM=015881399869  ATTNY_FLG=0  LITIG_FLG=0  SEV_BUCKET=0
  [long-attorney-lag     ] CVG=797880932  CLM_NUM=893186032080  ATTNY_FLG=1  LITIG_FLG=1  SEV_BUCKET=2
  [long-attorney-lag     ] CVG=78205258  CLM_NUM=817847267147  ATTNY_FLG=1  LITIG_FLG=0  SEV_BUCKET=3
  [long-litigation-lag   ] CVG=155761698  CLM_NUM=552059972837  ATTNY_FLG=1  LITIG_FLG=1  SEV_BUCKET=3
  [long-litigation-lag   ] CVG=65799065  CLM_NUM=586628360161  ATTNY_FLG=1  LITIG_FLG=1  SEV_BUCKET=0
  [never-attorney        ] CVG=51330175  CLM_NUM=613627010798  ATTNY_FLG=0  LITIG_FLG=0  SEV_BUCKET=0
  [never-attorney        ] 


Loaded notes for 8 distinct claims


In [9]:
summary_rows = []
detailed_results = {}

for r, label in test_cases:
    claim_notes = test_notes_all[test_notes_all.CLAIM_NUM == r.CLM_NUM].reset_index(drop=True)
    for model in ['Attorney', 'Litigation', 'Severity']:
        result = build_snapshots(r, model, claim_notes)
        detailed_results[(r.CLM_CVG_PK, model)] = result
        day_progression = ','.join(str(x['SNAPSHOT_DAY']) for x in result)
        n_progression = ','.join(str(x['N_NOTES']) for x in result)
        summary_rows.append({
            'scenario': label, 'CLM_CVG_PK': r.CLM_CVG_PK, 'model': model,
            'n_snapshots': len(result), 'days': day_progression, 'n_notes': n_progression,
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,scenario,CLM_CVG_PK,model,n_snapshots,days,n_notes
0,sibling-divergent,64914524,Attorney,1,1,1
1,sibling-divergent,64914524,Litigation,1,1,1
2,sibling-divergent,64914524,Severity,10,"1,90,180,270,365,540,730,900,1095,1460","1,19,29,36,41,55,60,61,63,73"
3,sibling-divergent,72105218,Attorney,8,"1,7,15,30,60,90,180,365","1,3,5,9,15,19,29,41"
4,sibling-divergent,72105218,Litigation,10,"1,90,180,270,365,540,730,900,1095,1460","1,19,29,36,41,55,60,61,63,73"
5,sibling-divergent,72105218,Severity,10,"1,90,180,270,365,540,730,900,1095,1460","1,19,29,36,41,55,60,61,63,73"
6,sibling-divergent,51064600,Attorney,1,1,1
7,sibling-divergent,51064600,Litigation,1,1,1
8,sibling-divergent,51064600,Severity,10,"1,90,180,270,365,540,730,900,1095,1460","1,19,29,36,41,55,60,61,63,73"
9,sibling-divergent,23041189,Attorney,1,1,1


---
## Full-scale build: architecture

Scaling `build_snapshots` from the 12 test cases above to all **167,726 closed coverages**, for all 3
models, split into stratified train/val/holdout, writing to S3 — without needing to hold the entire
output in memory/local disk at once, and without re-scanning the raw `BI_Notes` CSVs over and over.

**Resource check done before deciding this plan:** 88GB free disk, 54GB available RAM, 16 CPU cores;
the raw `BI_Notes` corpus is only 3.2GB total. So the *input* comfortably fits in memory — the real risk
is the *output*, since cumulative `NOTE_TEXT` duplicates earlier notes across every snapshot row of a
CVG, so total output size can be several times the input. That's what batching protects against, not
the input read.

**Plan:**
1. **Scan `BI_Notes` once, cache it** to a local parquet (`CLAIM_NUM`/`DATE`/`TEXT_NOTE` only) — this is
   the expensive step (~140s), done exactly **once**, shared across all 3 models (not re-scanned per
   model or per batch).
2. **Assign one global stratified train/val/holdout split, once**, over the full closed-coverage
   universe — stratified on `ORG_GROUP_NAME` + `POL_STATE` + `LOSS_YEAR` + all three rolled-up targets
   together (`ATTNY_FLG`, `LITIG_FLG`, `SEV_BUCKET`), so a given `CLM_NUM` lands in the same split for
   every model (see the dedicated split cell).
3. **One reusable function**, `build_dataset_for_model(model, notes_all, closed_df, split_map,
   batch_size_claims)` — same function body, called separately for `'Attorney'`, `'Litigation'`,
   `'Severity'`, each time filtering the shared global split down to that model's relevant claims
   rather than recomputing its own split.
4. Inside it: filter `closed_df` to the claims relevant for that model, then **group by `CLM_NUM` and
   batch on whole claims within each split** (never split one claim's sibling CVGs across two batches,
   or across two splits — a batch is a set of complete claims from one split, each with all of its
   closed CVGs processed together). For each batch: build every CVG's snapshots, write the batch's rows
   to a local parquet file, upload it to
   `{S3_BUCKET}/{S3_PREFIX}/{model}/{split}/`, delete the local file, and
   clear the batch's rows from memory before moving to the next batch. The shared `notes_all` stays
   resident in memory throughout (never reloaded or cleared) — only the per-batch *output* is churned.
5. **Run all 3 models in parallel** as separate OS processes (not threads — this is CPU-bound work, so
   the GIL would block real concurrency within one process). With 16 cores and no real memory pressure
   from 3 concurrent copies of the (small) cached notes parquet, this should run close to 3x faster than
   sequential. Each process writes its own progress to a **separate log file**, so even though they run
   concurrently, each model's build is cleanly reviewable on its own afterward.

### Output layout — S3 paths and parquet schema

**S3 layout** — one prefix per `(model, split)`, one parquet part-file per batch (part-files, not a
single file, since each batch is written+uploaded+cleared independently):

```
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/
├── Attorney/
│   ├── train/
│   │   ├── attorney_batch_0000.parquet
│   │   └── ...
│   ├── val/
│   │   └── attorney_batch_0000.parquet
│   └── holdout/
│       └── attorney_batch_0000.parquet
├── Litigation/
│   ├── train/ ...
│   ├── val/ ...
│   └── holdout/ ...
└── Severity/
    ├── train/ ...
    ├── val/ ...
    └── holdout/ ...
```

**Parquet schema — one row per `(CLM_CVG_PK, snapshot)`:**

| Column | Type | Meaning |
|---|---|---|
| `CLM_CVG_PK` | int64 | Coverage identifier — join key back to `BI-Coverage-ALL` |
| `CLM_NUM` | string | Claim identifier (kept for traceability; multiple `CLM_CVG_PK`s can share one) |
| `SNAPSHOT_NUM` | int | 1, 2, 3... — sequential snapshot index *within this CVG* |
| `SNAPSHOT_DAY` | int | which day-grid cutoff (`CLM_OPEN_DT + day`) this snapshot corresponds to |
| `N_NOTES` | int | cumulative note count included as of this snapshot |
| `NOTE_TEXT` | string | the full cumulative note history up to this snapshot (`SNAPSHOT (k)` + `NOTE n` markers) |
| `TARGET` (Attorney/Litigation only) | int (0/1) | that CVG's `ATTNY_FLG`/`LITIG_FLG` — constant across all its rows |
| `TARGET_BUCKET` (Severity only) | int (0-4) | that CVG's `SEV_BUCKET` — constant across all its rows |

Note the label column name itself differs by model (`TARGET` vs. `TARGET_BUCKET`) — this matches
`TARGET_COL` in the code, not a typo. The train/val/holdout **split itself is not a column** in the
parquet — it's encoded entirely by which `{split}/` folder the file lives in. Row count per
`(model, split)` is **not** fixed in advance — it depends on how many day-grid points survive the
boundary-stop + dedup + 0-note-skip rules per CVG (see the pseudocode at the top) and on how many claims the
stratified split assigns to each bucket, so expect noticeably different total row counts across all
nine `(model, split)` combinations.

### Step 1 — cache all notes once (local parquet, shared by all 3 models)

In [10]:
import time

NOTES_CACHE_PATH = str(TMP_DIR / 'all_notes_cache.parquet')

t0 = time.time()
parts = []
for f in note_files:
    df = pd.read_csv(f, usecols=['CLAIM_NUM', 'DATE', 'TEXT_NOTE'], dtype={'CLAIM_NUM': str})  # reads local data
    df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
    parts.append(df)

all_notes = pd.concat(parts, ignore_index=True)
all_notes = all_notes.sort_values(['CLAIM_NUM', 'DATE']).reset_index(drop=True)
all_notes.to_parquet(NOTES_CACHE_PATH)

print(f'Cached {len(all_notes):,} notes ({all_notes.CLAIM_NUM.nunique():,} distinct claims) '
      f'to {NOTES_CACHE_PATH} in {time.time() - t0:.1f}s')
print(f'Cache file size: {os.path.getsize(NOTES_CACHE_PATH) / 1e6:.1f} MB')

Cached 4,645,819 notes (138,233 distinct claims) to /tmp/all_notes_cache.parquet in 45.5s
Cache file size: 1581.4 MB


### Step 2 — the reusable `build_dataset_for_model` function

Reuses `build_snapshots` / `eligible_note_count` / `clean_note` already defined above. Batches on
whole claims, writes + uploads + clears per batch.

### Stratified claim-level train/val/holdout split (70/15/15) — global, shared across all 3 models

Computed **once**, over the full closed-coverage universe (not filtered per model), and then reused
by Attorney/Litigation/Severity alike — so a given `CLM_NUM` lands in the same split for all three
models. Stratifies on `ORG_GROUP_NAME` +
`POL_STATE` + `LOSS_YEAR` + **all three** rolled-up targets together (`ATTNY_FLG`, `LITIG_FLG`,
`SEV_BUCKET`), assigned **once per `CLM_NUM`** (not per `CLM_CVG_PK`) — sibling coverages of one claim
share note text, so letting them land in different splits would leak that shared text across the
split boundary. For a claim with multiple coverages, the 3 claim-level attributes are identical
across siblings (they are policy/claim-level, not coverage-level), and each target is rolled up to `max(target)` across the claim's
coverages (same convention as the severity roll-up in `01`). `SEV_BUCKET` keeps its full 5
levels (0-4) in the stratification key rather than being collapsed — this gives 956 strata, a few of which
fall into the "<3 claims" fallback below, but it is the most faithful stratification of the three targets together.
Strata with fewer than 3 claims can't be meaningfully split 3 ways — those are all assigned to
`train`.

In [11]:
def assign_global_claim_splits(closed_df, seed=42, train_frac=0.70, val_frac=0.15):
    """closed_df: the full closed-coverage universe (not filtered per model).
    Stratifies once per CLM_NUM on ORG_GROUP_NAME + POL_STATE + LOSS_YEAR + all three rolled-up
    targets (ATTNY_FLG, LITIG_FLG, SEV_BUCKET), so every model reuses the exact same split -- a
    claim lands in the same train/val/holdout bucket for Attorney, Litigation, and Severity alike.
    Returns a pd.Series indexed by CLM_NUM with values 'train' / 'val' / 'holdout'.
    """
    claim_attrs = closed_df.groupby('CLM_NUM').agg(
        ORG_GROUP_NAME=('ORG_GROUP_NAME', 'first'),
        POL_STATE=('POL_STATE', 'first'),
        LOSS_YEAR=('LOSS_YEAR', 'first'),
        _ATTNY=('ATTNY_FLG', 'max'),
        _LITIG=('LITIG_FLG', 'max'),
        _SEV=('SEV_BUCKET', 'max'),
    )
    claim_attrs['STRATUM'] = (
        claim_attrs['ORG_GROUP_NAME'].astype(str) + '|' +
        claim_attrs['POL_STATE'].astype(str) + '|' +
        claim_attrs['LOSS_YEAR'].astype(str) + '|' +
        claim_attrs['_ATTNY'].astype(str) + '|' +
        claim_attrs['_LITIG'].astype(str) + '|' +
        claim_attrs['_SEV'].astype(str)
    )

    rng = np.random.default_rng(seed)
    result = pd.Series(index=claim_attrs.index, dtype=object)

    for _, group in claim_attrs.groupby('STRATUM'):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        if n < 3:
            result.loc[idx] = 'train'
            continue
        n_train = max(1, round(n * train_frac))
        n_val = max(1, round(n * val_frac))
        if n_train + n_val >= n:
            n_val = max(1, n - n_train - 1)
        result.loc[idx[:n_train]] = 'train'
        result.loc[idx[n_train:n_train + n_val]] = 'val'
        result.loc[idx[n_train + n_val:]] = 'holdout'

    return result, claim_attrs['STRATUM']


# single global split, computed once over the full closed-coverage universe, reused by all 3 models
GLOBAL_SPLIT_MAP, GLOBAL_STRATUM = assign_global_claim_splits(closed)
print('Global split sizes (claims):')
print(GLOBAL_SPLIT_MAP.value_counts())
print(f'\nfractions: {(GLOBAL_SPLIT_MAP.value_counts(normalize=True) * 100).round(1).to_dict()}')
print(f'\ndistinct strata: {GLOBAL_STRATUM.nunique():,} '
      f'(strata with < 3 claims, forced to train: {(GLOBAL_STRATUM.value_counts() < 3).sum():,})')

Global split sizes (claims):
train      81212
val        17418
holdout    17267
Name: count, dtype: int64

fractions: {'train': 70.1, 'val': 15.0, 'holdout': 14.9}

distinct strata: 956 (strata with < 3 claims, forced to train: 146)


In [12]:
import boto3

LOCAL_OUT_DIR = str(TMP_DIR / 'dataset_build')
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)


def build_dataset_for_model(model, notes_all, closed_df, split_map, batch_size_claims=5000, log=print):
    relevant = closed_df.dropna(subset=[BOUNDARY_COL[model]])

    model_split_map = split_map.loc[split_map.index.intersection(relevant['CLM_NUM'].unique())]
    split_counts = model_split_map.value_counts()
    log(f'[{model}] {len(relevant):,} relevant closed coverages across {len(model_split_map):,} claims | '
        f'split sizes (claims): train={split_counts.get("train", 0):,} '
        f'val={split_counts.get("val", 0):,} holdout={split_counts.get("holdout", 0):,}')

    s3 = boto3.client('s3')
    total_rows = 0
    target_col = TARGET_COL[model]

    for split_name in ['train', 'val', 'holdout']:
        split_claims = model_split_map[model_split_map == split_name].index.to_numpy()
        n_batches = (len(split_claims) + batch_size_claims - 1) // batch_size_claims

        for batch_idx in range(n_batches):
            t0 = time.time()
            batch_claims = split_claims[batch_idx * batch_size_claims:(batch_idx + 1) * batch_size_claims]
            batch_claims_set = set(batch_claims)

            batch_notes = notes_all[notes_all.CLAIM_NUM.isin(batch_claims_set)]
            batch_cvgs = relevant[relevant.CLM_NUM.isin(batch_claims_set)]

            batch_rows = []
            for cn, cn_notes in batch_notes.groupby('CLAIM_NUM', sort=False):
                cn_rows = batch_cvgs[batch_cvgs.CLM_NUM == cn]
                if len(cn_rows) == 0:
                    continue
                cn_notes_sorted = cn_notes.sort_values('DATE').reset_index(drop=True)
                for _, cvg_row in cn_rows.iterrows():
                    batch_rows.extend(build_snapshots(cvg_row, model, cn_notes_sorted))

            if batch_rows:
                batch_df = pd.DataFrame(batch_rows)
                local_path = os.path.join(LOCAL_OUT_DIR, f'{model.lower()}_{split_name}_batch_{batch_idx:04d}.parquet')
                batch_df.to_parquet(local_path, index=False)

                s3_key = f'{S3_PREFIX}/{model}/{split_name}/{model.lower()}_batch_{batch_idx:04d}.parquet'
                s3.upload_file(local_path, S3_BUCKET, s3_key)  # S3 write
                os.remove(local_path)

                total_rows += len(batch_df)
                n_cvgs = batch_df.CLM_CVG_PK.nunique()
                avg_notes = batch_df.N_NOTES.mean()
                med_notes = batch_df.N_NOTES.median()
                avg_text_len = batch_df.NOTE_TEXT.str.len().mean()
                snapshots_per_cvg = len(batch_df) / n_cvgs
                target_mean = batch_df[target_col].mean()

                log(f'[{model}/{split_name}] batch {batch_idx + 1}/{n_batches}: {len(batch_claims)} claims -> '
                    f'{len(batch_df)} rows ({n_cvgs} CVGs) -> s3://{S3_BUCKET}/{s3_key} ({time.time() - t0:.1f}s)')
                log(f'    EDA: avg_notes/snapshot={avg_notes:.1f} median_notes={med_notes:.0f} '
                    f'avg_text_len={avg_text_len:.0f}chars snapshots/CVG={snapshots_per_cvg:.1f} '
                    f'mean_{target_col}={target_mean:.3f}')
                del batch_df
            else:
                log(f'[{model}/{split_name}] batch {batch_idx + 1}/{n_batches}: {len(batch_claims)} claims -> 0 rows, skipped')

            del batch_rows, batch_notes, batch_cvgs

    log(f'[{model}] DONE. total rows written: {total_rows:,}')
    return total_rows

### Step 3 — launch all 3 models in parallel (separate processes, separate logs)

`multiprocessing.Process` with the default `fork` start method (Linux) — each child inherits the
already-loaded `notes_all`/`closed` via copy-on-write, so there's no re-read or re-pickling of the
cached notes. Non-blocking: this cell just launches and returns immediately; use the monitor cell below
to check progress via each model's log file.

In [13]:
import multiprocessing

LOG_DIR = str(TMP_DIR / 'dataset_build_logs')
os.makedirs(LOG_DIR, exist_ok=True)


def _worker(model, notes_all_, closed_df_, split_map_, batch_size, log_path):
    with open(log_path, 'w') as f:
        def log(msg):
            line = f'{time.strftime("%H:%M:%S")} {msg}'
            f.write(line + '\n')
            f.flush()
        try:
            build_dataset_for_model(model, notes_all_, closed_df_, split_map_, batch_size_claims=batch_size, log=log)
            log(f'[{model}] EXIT_OK')
        except Exception as e:
            log(f'[{model}] EXIT_ERROR: {type(e).__name__}: {e}')
            raise


notes_all = pd.read_parquet(NOTES_CACHE_PATH)  # reads local data

build_processes = []
for model in ['Attorney', 'Litigation', 'Severity']:
    log_path = os.path.join(LOG_DIR, f'{model.lower()}.log')
    p = multiprocessing.Process(target=_worker, args=(model, notes_all, closed, GLOBAL_SPLIT_MAP, 5000, log_path))
    p.start()
    build_processes.append((model, p, log_path))
    print(f'Launched {model} build: PID={p.pid}, log={log_path}')

print('\nAll 3 processes launched (non-blocking). Use the monitor cell to check progress.')

Launched Attorney build: PID=1539, log=/tmp/dataset_build_logs/attorney.log


Launched Litigation build: PID=1540, log=/tmp/dataset_build_logs/litigation.log


Launched Severity build: PID=1541, log=/tmp/dataset_build_logs/severity.log

All 3 processes launched (non-blocking). Use the monitor cell to check progress.


### Monitor cell — check progress / completion (safe to re-run anytime)

In [14]:
for model, p, log_path in build_processes:
    alive = p.is_alive()
    status = 'RUNNING' if alive else f'FINISHED (exitcode={p.exitcode})'
    print(f'=== {model}  [{status}] ===')
    if os.path.exists(log_path):
        with open(log_path) as f:
            lines = f.readlines()
        for line in lines[-5:]:
            print('  ' + line.rstrip())
    print()

=== Attorney  [RUNNING] ===
  01:55:48 [Attorney] 167,726 relevant closed coverages across 115,897 claims | split sizes (claims): train=81,212 val=17,418 holdout=17,267

=== Litigation  [RUNNING] ===
  01:55:48 [Litigation] 167,719 relevant closed coverages across 115,895 claims | split sizes (claims): train=81,211 val=17,418 holdout=17,266

=== Severity  [RUNNING] ===



### Wait for completion — block until all 3 builds finish, then summarize what landed in S3

In [15]:
for model, p, log_path in build_processes:
    p.join()
    print(f'{model}: exitcode={p.exitcode}')
    with open(log_path) as f:
        print('  ' + f.readlines()[-2].rstrip())

s3 = boto3.client('s3')
for model in ['Attorney', 'Litigation', 'Severity']:
    for split_name in ['train', 'val', 'holdout']:
        objs = []
        for page in s3.get_paginator('list_objects_v2').paginate(Bucket=S3_BUCKET, Prefix=f'{S3_PREFIX}/{model}/{split_name}/'):  # S3 read
            objs.extend(page.get('Contents', []))
        print(f's3://{S3_BUCKET}/{S3_PREFIX}/{model}/{split_name}/: {len(objs)} files, '
              f'{sum(o["Size"] for o in objs) / 1e6:,.1f} MB')


Attorney: exitcode=0
  02:02:58 [Attorney] DONE. total rows written: 466,090


Litigation: exitcode=0
  02:04:46 [Litigation] DONE. total rows written: 388,956


Severity: exitcode=0
  02:09:23 [Severity] DONE. total rows written: 619,029
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Attorney/train/: 17 files, 838.4 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Attorney/val/: 4 files, 181.7 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Attorney/holdout/: 4 files, 176.0 MB


s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Litigation/train/: 17 files, 1,478.0 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Litigation/val/: 4 files, 326.5 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Litigation/holdout/: 4 files, 308.5 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Severity/train/: 16 files, 3,459.8 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Severity/val/: 4 files, 754.4 MB
s3://bi-severity-capstone/adjuster-notes/manual-feature-eng/datasets/Severity/holdout/: 4 files, 729.3 MB
